In [15]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import boto3
from botocore import UNSIGNED
from botocore.client import Config
from tqdm import tqdm

# Initialize the S3 client (unsigned for public NOAA Open Data bucket)
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

BUCKET_NAME = "noaa-nesdis-tcprimed-pds"

#### Downloading TC Primed

In [5]:
tc_primed = pd.read_csv('./tropical-cyclones/matched-tcprimed-[2023-2025].csv')

In [6]:
tc_primed.head()

,year,basin,storm_number,storm_id,satellite,time,size,file_path,product
0,2023,AL,1,AL012023,AMSR2,20230115070023,0.013367,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
1,2023,AL,1,AL012023,AMSR2,20230115180333,0.013604,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
2,2023,AL,1,AL012023,AMSR2,20230116060438,0.014802,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
3,2023,AL,1,AL012023,AMSR2,20230116170836,0.015796,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
4,2023,AL,1,AL012023,AMSR2,20230117064610,0.013443,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final


In [7]:
selected_storms = ['AL032023', 'AL042023', 'AL052023']

In [9]:
tc_primed[tc_primed['storm_id'].isin(selected_storms)]

,year,basin,storm_number,storm_id,satellite,time,size,file_path,product
66,2023,AL,3,AL032023,AMSR2,20230619155911,0.013670,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
67,2023,AL,3,AL032023,AMSR2,20230620041426,0.013944,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
68,2023,AL,3,AL032023,AMSR2,20230620164243,0.012700,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
69,2023,AL,3,AL032023,AMSR2,20230621045721,0.012724,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
70,2023,AL,3,AL032023,AMSR2,20230621172604,0.012921,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
...,...,...,...,...,...,...,...,...,...
536,2023,AL,5,AL052023,SSMIS,20230724072953,0.005524,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
537,2023,AL,5,AL052023,SSMIS,20230724171732,0.006024,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
538,2023,AL,5,AL052023,SSMIS,20230725071522,0.005504,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final
539,2023,AL,5,AL052023,SSMIS,20230725170451,0.003301,noaa-nesdis-tcprimed-pds/v01r01/final/2023/AL/...,final


In [19]:
storm = "AL052023"

local_path = f"/home/users/annaju/data/tc-primed/{storm}"
os.makedirs(local_path, exist_ok=True)

tc_primed_selected = tc_primed[tc_primed["storm_id"] == storm]

print("Number of files to download:", len(tc_primed_selected))

seen_names = {}

for fp in tqdm(tc_primed_selected["file_path"]):
    fp = str(fp).strip()

    # Accept any of these formats in file_path:
    # 1) s3://bucket/key
    # 2) bucket/key
    # 3) key
    if fp.startswith("s3://"):
        no_scheme = fp[5:]
        bucket, key = no_scheme.split("/", 1)
    elif "/" in fp and fp.split("/", 1)[0] == BUCKET_NAME:
        bucket = BUCKET_NAME
        key = fp.split("/", 1)[1]
    else:
        bucket = BUCKET_NAME
        key = fp

    # Flatten directory structure: keep only the filename locally
    base_name = os.path.basename(key)
    count = seen_names.get(base_name, 0)
    seen_names[base_name] = count + 1

    if count == 0:
        local_name = base_name
    else:
        stem, ext = os.path.splitext(base_name)
        local_name = f"{stem}_{count}{ext}"

    local_file_path = os.path.join(local_path, local_name)

    # Download the file
    s3.download_file(bucket, key, local_file_path)

print("Download completed")

Number of files to download: 310


100%|██████████| 310/310 [02:36<00:00,  1.98it/s]

Download completed
